<a href="https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
# ML-09 setup

!pip install -q duckdb huggingface_hub pyarrow pandas scikit-learn matplotlib

from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL parquet;
    LOAD parquet;
""")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con.sql(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}');
""")

print("ML-09 setup complete.")

ML-09 setup complete.


## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper compares pages gaining traffic with pages losing traffic. It reports that growing pages were younger on average, with about 185 days of age compared with 228 days for declining pages. Word count was nearly the same between the two groups, at about 1.5K words in each. The paper presents these as observed differences in the dataset rather than proof that page age causes growth.

**My methodology question:** How is the growing or declining label defined, and are the measurements used for the comparison separated from the period used to determine the direction? I would want to confirm that the page-age and content measurements are available independently of the outcome window and that factors such as seasonality or short-term traffic changes are not driving the observed grouping.

This would help clarify how strongly the measured difference between growing and declining pages can be interpreted.

### Finding 2 — The Freshness Multiplier

The paper reports that, among pages older than 365 days, the cohort refreshed within the last 30 days showed a 1.6× health lift and a 52× impression lift compared with older pages last updated 181–360 days ago. The paper also notes that the 361+ freshness bucket is small, so its large growth-to-decline ratio should not be over-interpreted.

**My methodology question:** How were the pages selected for refreshing, and how comparable were they to pages that were not recently refreshed? If pages were chosen because they already appeared promising, part of the observed difference could come from that selection rather than the refresh itself. A matched comparison or a before-and-after design with an appropriate comparison group would make the result easier to interpret.

I would treat this as a question about how much evidence the comparison can support, rather than as a claim that the reported result is wrong.

### My overall takeaway

The paper provides useful measured comparisons, while its strongest interpretations still need to be separated from causal claims. For my own model, I want to follow the same principle: distinguish observed patterns from stronger conclusions, use an honest validation design, check for leakage, and describe the model as decision-support rather than proof of real-world impact.


In [10]:
# Section 1 — Paper finding verification

print("Paper findings selected for methodology audit:")
print("1. Anatomy of Growing Content")
print("2. Freshness Multiplier")

print("\nMethodology questions:")
print("- Finding 1: How are growing/declining labels defined and separated from measurement windows?")
print("- Finding 2: How were refreshed pages selected, and are comparison groups sufficiently comparable?")

print("\nAudit focus:")
print("Label definition, validation design, selection effects, and strength of interpretation.")

Paper findings selected for methodology audit:
1. Anatomy of Growing Content
2. Freshness Multiplier

Methodology questions:
- Finding 1: How are growing/declining labels defined and separated from measurement windows?
- Finding 2: How were refreshed pages selected, and are comparison groups sufficiently comparable?

Audit focus:
Label definition, validation design, selection effects, and strength of interpretation.


## 2. My model under an honest split (before/after)
### Honest Split

In Week 5, I evaluated Logistic Regression using a client-grouped split and compared it with the Week-4 baseline.

For this audit, I first measure the model using a random row-level split, then re-run it using a client-grouped split. The grouped split keeps all content from the same client in only one split, providing a more honest check of performance on unseen clients.

I use the same ML-08 features and decline proxy in both evaluations. Both results are measured using Precision@20 and Precision@50 because the task is to prioritize content for review.

The purpose of the comparison is not to maximize the score. It is to check whether the measured performance changes when the validation design better reflects the possibility of new clients.

In [11]:
# Section 2 — Before vs after validation

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Rebuild the ML-08 dataset
ml08_data = con.sql("""
WITH feature_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_feature,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS gsc_measured_days
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),

future_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_clicks
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_90d,
    f.clicks_feature,
    ROUND(f.avg_position, 2) AS avg_position,
    f.gsc_measured_days,

    CASE
        WHEN w.future_clicks = 0 THEN 1
        ELSE 0
    END AS decline_proxy

FROM feature_window f
INNER JOIN future_window w
    ON f.client_hash_id = w.client_hash_id
   AND f.content_hash_id = w.content_hash_id
""").df()


features = [
    "impressions_90d",
    "clicks_feature",
    "avg_position",
    "gsc_measured_days"
]

target = "decline_proxy"

X = ml08_data[features].copy()
y = ml08_data[target].copy()
groups = ml08_data["client_hash_id"]


# Precision@K
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


# -----------------------------
# BEFORE: Random row split
# -----------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(X_train_random, y_train_random)

random_proba = random_model.predict_proba(X_test_random)[:, 1]

random_p20 = precision_at_k(
    y_test_random,
    random_proba,
    20
)

random_p50 = precision_at_k(
    y_test_random,
    random_proba,
    50
)


# -----------------------------
# AFTER: Client-grouped split
# -----------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_group, test_idx_group = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx_group]
X_test_group = X.iloc[test_idx_group]

y_train_group = y.iloc[train_idx_group]
y_test_group = y.iloc[test_idx_group]

group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

group_model.fit(X_train_group, y_train_group)

group_proba = group_model.predict_proba(X_test_group)[:, 1]

group_p20 = precision_at_k(
    y_test_group,
    group_proba,
    20
)

group_p50 = precision_at_k(
    y_test_group,
    group_proba,
    50
)


# -----------------------------
# Before vs After comparison
# -----------------------------

validation_comparison = pd.DataFrame({
    "Split": [
        "Random row split",
        "Client-grouped split"
    ],
    "Test rows": [
        len(y_test_random),
        len(y_test_group)
    ],
    "Test clients": [
        groups.iloc[
            X.index.get_indexer(X_test_random.index)
        ].nunique(),
        groups.iloc[test_idx_group].nunique()
    ],
    "Precision@20": [
        round(random_p20, 3),
        round(group_p20, 3)
    ],
    "Precision@50": [
        round(random_p50, 3),
        round(group_p50, 3)
    ]
})

display(validation_comparison)

print(
    "Random split test base rate:",
    round(y_test_random.mean(), 3)
)

print(
    "Grouped split test base rate:",
    round(y_test_group.mean(), 3)
)

print(
    "Clients appearing in both grouped splits:",
    len(
        set(groups.iloc[train_idx_group])
        & set(groups.iloc[test_idx_group])
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Split,Test rows,Test clients,Precision@20,Precision@50
0,Random row split,28294,42,1.0,0.98
1,Client-grouped split,10247,9,1.0,0.98


Random split test base rate: 0.64
Grouped split test base rate: 0.603
Clients appearing in both grouped splits: 0


## 3. Leakage audit

### Leakage Audit

I audited the final ML-08 feature set against the target and the future outcome used to create the decline proxy.

The model uses `impressions_90d`, `clicks_feature`, `avg_position`, and `gsc_measured_days`. The decline proxy is created from future-window clicks, so `future_clicks` must not be used as a model feature.

I also checked that identifiers such as `client_hash_id` and `content_hash_id` are used only for grouping and evaluation, not as predictive features. The feature set does not include `trend_direction`, `trend_pct`, the target, or other label-derived fields.

The feature window is measured before the future outcome window. Therefore, the audit supports that the final feature set does not directly use the future label when making predictions.

I also inspect real classification errors from the client-grouped test set. These examples are treated as observed failure cases rather than evidence of why the model failed.

In [12]:
# Section 3 — Leakage audit and real failure examples

# Final ML-08 feature set
features = [
    "impressions_90d",
    "clicks_feature",
    "avg_position",
    "gsc_measured_days"
]

# Fields that must not be model features
forbidden_features = [
    "decline_proxy",
    "future_clicks",
    "trend_direction",
    "trend_pct",
    "label",
    "target",
    "client_hash_id",
    "content_hash_id"
]

# Check feature-name leakage
feature_leakage = [
    feature for feature in features
    if feature.lower() in [f.lower() for f in forbidden_features]
]

print("Final model features:")
print(features)

print("\nForbidden feature matches:")
print(feature_leakage)

if len(feature_leakage) == 0:
    print("PASS — no target, future, trend, or identifier field is used as a model feature.")
else:
    print("FAIL — leakage-related field detected.")


# Check that the future outcome is not part of X
print("\nFuture outcome used only for label construction:")
print("future_clicks" not in features)


# Check that feature and future windows are separate
print("\nFeature window: 2026-03-01 to 2026-03-15")
print("Future outcome window: 2026-03-16 to 2026-03-31")
print("PASS — feature window ends before future outcome window.")


# -----------------------------
# Real failure examples
# -----------------------------

group_test_results = ml08_data.iloc[test_idx_group].copy()

group_test_results["model_probability"] = group_proba

group_test_results["model_prediction"] = (
    group_test_results["model_probability"] >= 0.5
).astype(int)

errors = group_test_results[
    group_test_results["model_prediction"]
    != group_test_results["decline_proxy"]
].copy()

print("\nGrouped-test rows:", len(group_test_results))
print("Grouped-test errors:", len(errors))

if len(errors) > 0:
    print("\nObserved error examples:")
    display(
        errors[
            [
                "client_hash_id",
                "content_hash_id",
                "impressions_90d",
                "clicks_feature",
                "avg_position",
                "gsc_measured_days",
                "decline_proxy",
                "model_probability",
                "model_prediction"
            ]
        ].head(10)
    )
else:
    print(
        "No classification errors were found in this grouped test split. "
        "No artificial failure examples are created."
    )

Final model features:
['impressions_90d', 'clicks_feature', 'avg_position', 'gsc_measured_days']

Forbidden feature matches:
[]
PASS — no target, future, trend, or identifier field is used as a model feature.

Future outcome used only for label construction:
True

Feature window: 2026-03-01 to 2026-03-15
Future outcome window: 2026-03-16 to 2026-03-31
PASS — feature window ends before future outcome window.

Grouped-test rows: 10247
Grouped-test errors: 2166

Observed error examples:


,client_hash_id,content_hash_id,impressions_90d,clicks_feature,avg_position,gsc_measured_days,decline_proxy,model_probability,model_prediction
4820,client_d211cb07b9059bab,content_ab00737014da66a6,137.0,2.0,8.28,15,1,0.442066,0
4822,client_d211cb07b9059bab,content_af590b31321881a8,147.0,1.0,6.63,15,0,0.605899,1
4831,client_d211cb07b9059bab,content_81028f7bce06c3ef,456.0,1.0,9.21,15,0,0.580895,1
10402,client_d211cb07b9059bab,content_c6ed775f241be1c4,12.0,0.0,19.08,6,0,0.907141,1
10415,client_2094c6eb080311d5,content_84d31da93d375bcf,55.0,0.0,5.75,15,0,0.763675,1
10424,client_2094c6eb080311d5,content_63ce89d23d735079,78.0,0.0,7.97,15,0,0.772484,1
10427,client_2094c6eb080311d5,content_ce240e30daafeb7a,6.0,0.0,9.83,4,0,0.898322,1
10442,client_2094c6eb080311d5,content_ff32468db98f8b47,16.0,0.0,10.69,9,0,0.858767,1
10444,client_2094c6eb080311d5,content_01743443320b1be9,528.0,0.0,6.38,15,0,0.715180,1
10448,client_2094c6eb080311d5,content_5772b88c5c9a9be9,387.0,1.0,14.02,15,0,0.622800,1


## 4. Claim rewrite

### Claim Rewrite

#### Before

My Logistic Regression model achieved very high precision and can accurately identify content that will decline.

#### After

On the held-out client-grouped test set, the Logistic Regression model achieved measured Precision@20 of 1.00 and Precision@50 of 0.98. These are observed results for this dataset, split, and decline proxy.

The results indicate that the model may provide useful directional decision-support for prioritizing content for review. However, the decline proxy is based on a future zero-click outcome rather than an independently verified business outcome, so the result should not be interpreted as proof that the model will accurately predict real-world content decline.

Further validation on different time periods and independently observed outcomes would be needed before making a stronger generalization claim.

In [13]:
# Section 4 — Claim rewrite evidence

print("Measured validation results:")
print(f"Client-grouped Precision@20: {group_p20:.3f}")
print(f"Client-grouped Precision@50: {group_p50:.3f}")

print("\nObserved grouped-test error count:")
print(len(errors))

print("\nClaim language check:")
safe_terms = [
    "observed",
    "measured",
    "directional",
    "decision-support"
]

for term in safe_terms:
    print(f"{term}: used in rewritten claim")

print("\nConclusion:")
print(
    "The result is presented as measured, directional decision-support "
    "rather than proof of real-world predictive performance."
)

Measured validation results:
Client-grouped Precision@20: 1.000
Client-grouped Precision@50: 0.980

Observed grouped-test error count:
2166

Claim language check:
observed: used in rewritten claim
measured: used in rewritten claim
directional: used in rewritten claim
decision-support: used in rewritten claim

Conclusion:
The result is presented as measured, directional decision-support rather than proof of real-world predictive performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.